In [1]:
import numpy as np

# =========================================================
# Function to read GMX-PC statistics files
# =========================================================
def read_gmxpc_statistics_file(filename):
    """
    Reads TNI GMX-PC 'Statistics of Counting' ASC files.
    Extracts:
        - average count (I)
        - standard deviation (sigma_I)
    """
    with open(filename, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    avg = None
    sigma = None

    for line in lines:
        if line.startswith("Average Count"):
            avg = float(line.split("=")[1])
        elif line.startswith("Standard Deviation"):
            sigma = float(line.split("=")[1])

    return avg, sigma


# =========================================================
# 1 — YOU WRITE THE FILE NAMES HERE
# =========================================================
file_R1  = "ml1stat.asc"      # half-moon 1
file_R2  = "ml2stat.asc"      # half-moon 2
file_R12 = "2lstat.asc"     # both together
file_BKG = "cnada.asc"    # background noise


# =========================================================
# 2 — Read the files
# =========================================================
avg1, sigma1 = read_gmxpc_statistics_file(file_R1)
avg2, sigma2 = read_gmxpc_statistics_file(file_R2)
avg12, sigma12 = read_gmxpc_statistics_file(file_R12)
avg_bkg, sigma_bkg = read_gmxpc_statistics_file(file_BKG)

# Each measurement is 20 seconds
T = 20.0

# Convert to rates
R1  = avg1  / T
R2  = avg2  / T
R12 = avg12 / T
Rb  = avg_bkg / T

sigma_R1  = sigma1  / T
sigma_R2  = sigma2  / T
sigma_R12 = sigma12 / T
sigma_Rb  = sigma_bkg / T

# =========================================================
# 3 — Subtract background
# =========================================================
R1_corr  = R1  - Rb
R2_corr  = R2  - Rb
R12_corr = R12 - Rb

sigma_R1_corr  = np.sqrt(sigma_R1**2  + sigma_Rb**2)
sigma_R2_corr  = np.sqrt(sigma_R2**2  + sigma_Rb**2)
sigma_R12_corr = np.sqrt(sigma_R12**2 + sigma_Rb**2)

print("\n=== CORRECTED RATES (counts/s) ===")
print(f"R1_corr  = {R1_corr:.3f} ± {sigma_R1_corr:.3f}")
print(f"R2_corr  = {R2_corr:.3f} ± {sigma_R2_corr:.3f}")
print(f"R12_corr = {R12_corr:.3f} ± {sigma_R12_corr:.3f}")


# =========================================================
# 4 — Dead time formula:
# TR = (R1 + R2 - R12) / (2 R1 R2)
# =========================================================
TR = (R1_corr + R2_corr - R12_corr) / (2 * R1_corr * R2_corr)

# =========================================================
# 5 — Propagation of uncertainty
# =========================================================
dTR_dR1  = (1 - (R12_corr / R1_corr)) / (2 * R2_corr * R1_corr)
dTR_dR2  = (1 - (R12_corr / R2_corr)) / (2 * R1_corr * R2_corr)
dTR_dR12 = -1 / (2 * R1_corr * R2_corr)

sigma_TR = np.sqrt(
    (dTR_dR1  * sigma_R1_corr)**2 +
    (dTR_dR2  * sigma_R2_corr)**2 +
    (dTR_dR12 * sigma_R12_corr)**2
)

print("\n=== DEAD TIME RESULT ===")
print(f"Dead time TR = ({TR:.3e} ± {sigma_TR:.3e}) s")


FileNotFoundError: [Errno 2] No such file or directory: 'cnada.asc'